In [ ]:
# !pip install --upgrade google-play-scraper
# !pip install pyqt5
# !pip install --upgrade beautifulsoup4
# !pip install pyqtwebengine
# !pip install pandas
# !pip install selenium

In [ ]:
#from google_play_scraper import collection
#from google_play_scraper.constants import Collection, Category

import time
from bs4 import BeautifulSoup
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google_play_scraper import app, permissions


#Identified risk categories in each app's permissions
risk_categories = {
    "overlay": ["draw over other apps"],
    "background": ["run at startup", "prevent device from sleeping"],
    "bluetooth": ["pair with Bluetooth devices", "access Bluetooth settings"],
    "network": ["full network access", "view network connections"]
}
PII_RISK_CATEGORIES = {
    "identity": [
        "read your contacts",
        "modify your contacts"
    ],
    "location": [
        "access precise location",
        "access approximate location"
    ],
    "persistent_identifier": [
        "read phone status and identity"
    ],
    "biometric": [
        "camera",
        "record audio"
    ],
    "storage_access": [
        "read the contents of your usb storage",
        "modify or delete the contents of your usb storage"
    ]
}
PII_TERMS = {
    "identity": [
        "email", "name", "phone number", "contact information"
    ],
    "location": [
        "location", "gps", "precise location"
    ],
    "persistent_identifier": [
        "device id", "advertising id", "imei", "ip address"
    ],
    "biometric": [
        "biometric", "facial recognition", "voice data", "fingerprint"
    ],
    "storage_access": [
        "files", "photos", "media", "storage"
    ]
}


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_policy(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "lxml")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        return soup.get_text("\n").lower()

    except Exception:
        return ""


def collect_app_data():
    error_apps = {}

    with open("family_apps_strict_enriched.json", "r") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    flag_dict = {}
    seen_apps = set()

    # --- Step 1: Prepare metadata first (no network yet) ---
    app_rows = []

    for _, row in df.iterrows():

        app_id = row.get("appId")
        if not app_id or app_id in seen_apps:
            continue

        seen_apps.add(app_id)

        url = row.get("privacyPolicy")
        if not url:
            continue

        app_rows.append({
            "app_id": app_id,
            "url": url,
            "title": row.get("title"),
            "genre": row.get("genre")
        })

    # --- Step 2: Parallelize policy fetching ---
    with ThreadPoolExecutor(max_workers=10) as executor:

        future_to_app = {
            executor.submit(fetch_policy, app["url"]): app
            for app in app_rows
        }

        for future in as_completed(future_to_app):

            app = future_to_app[future]
            app_id = app["app_id"]

            try:
                policy_text = future.result()
                perm = permissions(app_id)
                all_permissions = []
                for category in perm:
                  all_permissions.extend(perm[category])


                flag_dict[app_id] = {
                    "title": app["title"],
                    "genre": app["genre"],
                    "permissions": all_permissions,
                    "policy_text": policy_text
                }

            except Exception as e:
                print(f"Error processing {app_id}: {e}")
                error_apps[app_id] = e

    return flag_dict

In [ ]:
pii_gap_results = []
flag_dict = collect_app_data()
pii_permission_hits = 0

for app_id, data in flag_dict.items():
    perms_lower = [perm.lower() for perm in data["permissions"]]

    for category, perm_list in PII_RISK_CATEGORIES.items():
        if any(any(p.lower() in perm for perm in perms_lower) for p in perm_list):
            pii_permission_hits += 1
            break

print("Apps requesting at least one PII category:", pii_permission_hits)


for app_id, data in flag_dict.items():
    perms = data["permissions"]
    policy = data["policy_text"]

    if not policy or len(policy) < 500:
        continue

    perms_lower = [perm.lower() for perm in perms]

    for category, perm_list in PII_RISK_CATEGORIES.items():

        has_permission = any(
            any(p.lower() in perm for perm in perms_lower)
            for p in perm_list
        )

        mentions_pii = any(
            term in policy
            for term in PII_TERMS[category]
        )

        if has_permission and not mentions_pii:
            pii_gap_results.append({
                "app_id": app_id,
                "title": data["title"],
                "category": category
            })

unique_gap_apps = set(result["app_id"] for result in pii_gap_results)

print("Unique apps with at least one PII gap:", len(unique_gap_apps))
print("Total PII Gap Cases:", len(pii_gap_results))

## Cell Below may be adapted or deleted in the future. I'll keep it here for now. Right now it just checks for the keyword "bluetooth", but the cell below this one identifies key bluetooth phrases not just the word bluetooth.

# Task
Load the app data from "family_apps_strict_enriched.json" into a pandas DataFrame. Group the data by 'genre' and sample a total of 50-100 apps, ensuring a diverse mix of categories (e.g., Education, Games). Select the columns 'title' (rename to App Name), 'appId' (rename to Package ID), and 'privacyPolicy' (rename to Privacy Policy URL), then save this sampled dataframe to a CSV file named "sampled_apps_list.csv". Finally, confirm the file creation and display the first few rows of the sampled data.

## Load App Data

### Subtask:
Load the app data from 'family_apps_strict_enriched.json' into a pandas DataFrame.


**Reasoning**:
Load the JSON data into a pandas DataFrame as requested by the subtask instructions.



In [ ]:
import json
import pandas as pd

# Open the file and load the data
with open("family_apps_strict_enriched.json", "r") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.json_normalize(data)

# Display the first 5 rows and the shape
print("Shape of the DataFrame:", df.shape)
df.head()

In [ ]:
# Bert Test code
%pip install sentence-transformers transformers torch nltk
%pip install --user -U nltk

"""
Privacy Compliance Scanner
Detects obfuscated permission language in app listings and Terms of Service
using Sentence-BERT semantic similarity and zero-shot NLI classification.

Install dependencies:
    pip install sentence-transformers transformers torch nltk requests beautifulsoup4
"""

import numpy as np
import nltk
from sentence_transformers import SentenceTransformer
from transformers import pipeline
nltk.download('punkt_tab')
nltk.download("punkt", quiet=True)

# ── Anchor taxonomy ──────────────────────────────────────────────────────────

PERMISSION_ANCHORS = {
    "bluetooth_access": [
        "accessing nearby bluetooth devices",
        "nearby device discovery",
        "nearby device detection",
        "proximity sensing",
        "proximity detection",
        "peer-to-peer connectivity",
        "short-range wireless communication",
        "RF pairing",
        "radio frequency pairing",
        "device pairing",
        "wireless peripheral access",
        "local network discovery",
        "hardware handshake",
        "adjacent device scanning",
        "seamless connectivity",
        "smart device integration",
        "ecosystem pairing",
        "proximity-based features",
        "wireless sync",
    ],
    "location_tracking": [
        "tracking user location",
        "GPS access",
        "geolocation data",
        "real-time position tracking",
        "geographic data collection",
        "user whereabouts",
        "location history",
        "place detection",
        "movement tracking",
        "location-based services",
        "lat/long coordinates",
        "geofencing",
    ],
    "microphone_access": [
        "recording audio via microphone",
        "capturing voice input",
        "audio recording",
        "sound capture",
        "ambient noise detection",
        "voice activation",
        "acoustic sensing",
        "speech detection",
        "audio monitoring",
        "listening to user",
    ],
    "camera_access": [
        "accessing the camera",
        "capturing photos or video",
        "image capture",
        "visual input",
        "video recording",
        "photo access",
        "optical sensor access",
        "augmented reality input",
        "facial recognition input",
    ],
    "contact_access": [
        "reading user contacts",
        "address book access",
        "phone book data",
        "contact list retrieval",
        "friend list access",
        "social graph data",
        "user connections",
        "phonebook scanning",
    ],
    "storage_access": [
        "reading files from device",
        "accessing local storage",
        "file system access",
        "media library access",
        "external storage read",
        "document access",
        "photo library access",
        "disk read permission",
    ],
}

# ── Models ───────────────────────────────────────────────────────────────────

print("Loading models...")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
nli_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Models ready.\n")


# ── Precompute mean anchor embeddings ────────────────────────────────────────

def build_anchor_vectors(anchors: dict) -> dict:
    vectors = {}
    for permission, phrases in anchors.items():
        embs = sbert_model.encode(phrases)
        vectors[permission] = np.mean(embs, axis=0)
    return vectors

ANCHOR_VECTORS = build_anchor_vectors(PERMISSION_ANCHORS)


# ── Scoring ──────────────────────────────────────────────────────────────────

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def score_clause_sbert(clause: str, threshold: float = 0.50) -> list[dict]:
    """Return all permissions above threshold using SBERT cosine similarity."""
    emb = sbert_model.encode(clause)
    results = []
    for permission, anchor_vec in ANCHOR_VECTORS.items():
        score = cosine_similarity(emb, anchor_vec)
        if score >= threshold:
            results.append({"permission": permission, "score": round(score, 4), "method": "sbert"})
    return sorted(results, key=lambda x: x["score"], reverse=True)


def score_clause_nli(clause: str, threshold: float = 0.40) -> list[dict]:
    """Return all permissions above threshold using zero-shot NLI."""
    labels = list(PERMISSION_ANCHORS.keys())
    out = nli_classifier(clause, candidate_labels=labels)
    results = []
    for label, score in zip(out["labels"], out["scores"]):
        if score >= threshold:
            results.append({"permission": label, "score": round(score, 4), "method": "nli"})
    return results


# ── Document scanner ─────────────────────────────────────────────────────────

def scan_document(text: str, sbert_threshold=0.50, nli_threshold=0.40) -> list[dict]:
    """
    Split a document into sentences and flag any that match permission anchors.
    Returns a list of flagged findings.
    """
    sentences = nltk.sent_tokenize(text)
    findings = []

    for i, sentence in enumerate(sentences):
        sentence = sentence.strip()
        if len(sentence) < 20:  # skip very short fragments
            continue

        sbert_hits = score_clause_sbert(sentence, threshold=sbert_threshold)
        nli_hits   = score_clause_nli(sentence, threshold=nli_threshold)

        # Merge hits, preferring highest score per permission
        merged = {}
        for hit in sbert_hits + nli_hits:
            p = hit["permission"]
            if p not in merged or hit["score"] > merged[p]["score"]:
                merged[p] = hit

        if merged:
            findings.append({
                "sentence_index": i,
                "sentence": sentence,
                "flags": list(merged.values()),
            })

    return findings


def cross_reference(findings: list[dict], declared_permissions: list[str]) -> list[dict]:
    """
    Compare flagged permissions against officially declared permissions.
    Returns only findings where the permission is NOT declared.
    """
    declared_set = {p.lower() for p in declared_permissions}
    violations = []
    for finding in findings:
        undeclared_flags = [
            f for f in finding["flags"]
            if f["permission"] not in declared_set
        ]
        if undeclared_flags:
            violations.append({**finding, "flags": undeclared_flags})
    return violations


def print_report(findings: list[dict], title: str = "Scan Report"):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    if not findings:
        print("  No suspicious clauses found.\n")
        return
    for f in findings:
        print(f"\n  Sentence #{f['sentence_index']}")
        print(f"  \"{f['sentence']}\"")
        for flag in f["flags"]:
            print(f"    → {flag['permission']} (score={flag['score']}, method={flag['method']})")
    print()


# ── Example usage ────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # Simulated ToS text from a hypothetical app
    sample_tos = """
    Our app provides a seamless experience for all users.
    We may enable peer-to-peer proximity detection with surrounding hardware to improve connectivity.
    The app uses acoustic sensing to enhance voice-based features.
    Location-based services help us deliver personalized content.
    We access your social graph data to suggest relevant connections.
    The app may read files from device storage to enable media sharing.
    We are committed to protecting your privacy and data security.
    """

    # Permissions this app officially declared (e.g. from Play Store manifest)
    declared = ["location_tracking", "storage_access"]

    print("Scanning document...")
    all_findings = scan_document(sample_tos)
    print_report(all_findings, title="All Flagged Clauses")

    violations = cross_reference(all_findings, declared_permissions=declared)
    print_report(violations, title="Undeclared Permission Violations")

    # ── Scan a single clause interactively ──────────────────────────────────
    print("\n--- Single clause test ---")
    clause = "enables ecosystem pairing and wireless sync with adjacent hardware"
    print(f"Clause: \"{clause}\"")
    hits = score_clause_sbert(clause)
    for h in hits:
        print(f"  {h['permission']}: {h['score']} ({h['method']})")

In [ ]:
import ollama

def llama_check(permission, sentence):

    prompt = f"""
You are a privacy compliance analyzer.

Permission requested by the app:
{permission}

Policy sentence:
{sentence}

Question:
Does this sentence disclose the permission?

Answer YES or NO only.
"""

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return response["message"]["content"].strip()

In [ ]:
import ollama
from nltk.tokenize import sent_tokenize

print("\nScanning all apps...\n")

total_apps = len(flag_dict)
apps_with_findings = 0
total_violations = 0
all_results = []

for i, (app_id, data) in enumerate(flag_dict.items()):

    print(f"Scanning app {i+1}/{len(flag_dict)}: {data['title']}")

    title = data.get("title", "Unknown App")
    policy_text = data.get("policy_text", "")
    permissions = data.get("permissions", [])

    if not policy_text:
        continue

    findings = scan_document(policy_text)

    if findings:
        apps_with_findings += 1

    violations = cross_reference(findings, declared_permissions=permissions)

    if violations:

        sentences = sent_tokenize(policy_text)

        verified_violations = []

        for violation in violations:

            permission = violation.get("permission", "")

            disclosed = False

            for sentence in sentences:

                result = llama_check(permission, sentence)

                if "YES" in result.upper():
                    disclosed = True
                    break

            if not disclosed:
                verified_violations.append(violation)

        if verified_violations:

            total_violations += len(verified_violations)

            all_results.append({
                "app_id": app_id,
                "title": title,
                "violations": verified_violations
            })

            print_report(
                verified_violations,
                title=f"Undeclared Permissions Detected — {title}"
            )

print("\n==============================")
print("SCAN SUMMARY")
print("==============================")
print("Apps scanned:", total_apps)
print("Apps with suspicious clauses:", apps_with_findings)
print("Total undeclared permission violations:", total_violations)

In [ ]:
import ollama
from nltk.tokenize import sent_tokenize

